**Raw Baseline ELCS**

In [1]:
import pandas as pd
import numpy as np

from skeLCS.eLCS import eLCS

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

data_raw = pd.read_csv("../data/weatherAUS.csv")

print("Original shape:", data_raw.shape)
print(data_raw.head())


Original shape: (145460, 23)
         Date Location  MinTemp  MaxTemp  Rainfall  Evaporation  Sunshine  \
0  2008-12-01   Albury     13.4     22.9       0.6          NaN       NaN   
1  2008-12-02   Albury      7.4     25.1       0.0          NaN       NaN   
2  2008-12-03   Albury     12.9     25.7       0.0          NaN       NaN   
3  2008-12-04   Albury      9.2     28.0       0.0          NaN       NaN   
4  2008-12-05   Albury     17.5     32.3       1.0          NaN       NaN   

  WindGustDir  WindGustSpeed WindDir9am  ... Humidity9am  Humidity3pm  \
0           W           44.0          W  ...        71.0         22.0   
1         WNW           44.0        NNW  ...        44.0         25.0   
2         WSW           46.0          W  ...        38.0         30.0   
3          NE           24.0         SE  ...        45.0         16.0   
4           W           41.0        ENE  ...        82.0         33.0   

   Pressure9am  Pressure3pm  Cloud9am  Cloud3pm  Temp9am  Temp3pm  Ra

In [2]:
# remove rows where the target is missing
data_raw = data_raw.dropna(
    subset=["RainTomorrow"]
).copy()

# convert target to numeric
data_raw["RainTomorrow"] = data_raw["RainTomorrow"].map({
    "No": 0,
    "Yes": 1
})

# Date is removed for the raw baseline rather than feature engineered
data_raw = data_raw.drop(columns=["Date"])

print("Shape after removing missing target:", data_raw.shape)

print("\nTarget distribution:")
print(data_raw["RainTomorrow"].value_counts())

print("\nMissing feature values:")
print(data_raw.isna().sum().sort_values(ascending=False))


Shape after removing missing target: (142193, 22)

Target distribution:
RainTomorrow
0    110316
1     31877
Name: count, dtype: int64

Missing feature values:
Sunshine         67816
Evaporation      60843
Cloud3pm         57094
Cloud9am         53657
Pressure9am      14014
Pressure3pm      13981
WindDir9am       10013
WindGustDir       9330
WindGustSpeed     9270
WindDir3pm        3778
Humidity3pm       3610
Temp3pm           2726
WindSpeed3pm      2630
Humidity9am       1774
RainToday         1406
Rainfall          1406
WindSpeed9am      1348
Temp9am            904
MinTemp            637
MaxTemp            322
Location             0
RainTomorrow         0
dtype: int64


In [3]:
classlabel = "RainTomorrow"

x_raw = data_raw.drop(columns=[classlabel])
y_raw = data_raw[classlabel]

x_train, x_test, y_train, y_test = train_test_split(
    x_raw,
    y_raw,
    test_size=0.20,
    random_state=42,
    stratify=y_raw
)

print("Training:", x_train.shape)
print("Testing:", x_test.shape)

print("\nTraining target:")
print(y_train.value_counts())

print("\nTesting target:")
print(y_test.value_counts())


Training: (113754, 21)
Testing: (28439, 21)

Training target:
RainTomorrow
0    88252
1    25502
Name: count, dtype: int64

Testing target:
RainTomorrow
0    22064
1     6375
Name: count, dtype: int64


In [4]:
# categorical values must be numeric for eLCS
categorical_cols = x_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print("Categorical columns:")
print(categorical_cols)

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

x_train = x_train.copy()
x_test = x_test.copy()

x_train[categorical_cols] = encoder.fit_transform(
    x_train[categorical_cols]
)

x_test[categorical_cols] = encoder.transform(
    x_test[categorical_cols]
)


Categorical columns:
['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm', 'RainToday']


In [5]:
# tell eLCS which encoded columns are discrete categories
discrete_attributes = np.array([
    x_train.columns.get_loc(col)
    for col in categorical_cols
])

print("Discrete attribute indices:")
print(discrete_attributes)

# professor's eLCS requires numeric numpy arrays
x_train_np = x_train.to_numpy(dtype=float)
x_test_np = x_test.to_numpy(dtype=float)

y_train_np = y_train.to_numpy(dtype=float)
y_test_np = y_test.to_numpy(dtype=float)

print("\nTraining shape:", x_train_np.shape)
print("Testing shape:", x_test_np.shape)
print("Features:", x_train_np.shape[1])

print("\nMissing training feature values:",
      np.isnan(x_train_np).sum())

print("Missing testing feature values:",
      np.isnan(x_test_np).sum())


Discrete attribute indices:
[ 0  6  8  9 20]

Training shape: (113754, 21)
Testing shape: (28439, 21)
Features: 21

Missing training feature values: 253678
Missing testing feature values: 62881


In [6]:
# full raw eLCS baseline
# learning iterations match the number of training instances
raw_elcs = eLCS(
    learning_iterations=len(x_train_np),
    N=10000,
    discrete_attribute_limit="d",
    specified_attributes=discrete_attributes,
    random_state=42
)

raw_elcs.fit(
    x_train_np,
    y_train_np
)

print("Raw eLCS training complete")

raw_predictions = raw_elcs.predict(
    x_test_np
)

print(
    "Accuracy:",
    accuracy_score(
        y_test_np,
        raw_predictions
    )
)

print(
    "Balanced Accuracy:",
    balanced_accuracy_score(
        y_test_np,
        raw_predictions
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test_np,
        raw_predictions
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test_np,
        raw_predictions
    )
)


Raw eLCS training complete
Accuracy: 0.8294595449910335
Balanced Accuracy: 0.6384023411395015

Classification Report:
              precision    recall  f1-score   support

         0.0       0.83      0.98      0.90     22064
         1.0       0.85      0.29      0.43      6375

    accuracy                           0.83     28439
   macro avg       0.84      0.64      0.67     28439
weighted avg       0.83      0.83      0.80     28439


Confusion Matrix:
[[21727   337]
 [ 4513  1862]]


In [ ]:
# ============================================================
# SAVE RAW eLCS PREDICTIONS
# ============================================================

from pathlib import Path

results_dir = Path("../results")
results_dir.mkdir(parents=True, exist_ok=True)

raw_results = pd.DataFrame({
    "row_id": x_test.index.to_numpy(),
    "actual": y_test.to_numpy(),
    "prediction": raw_predictions
})

raw_results.to_csv(
    results_dir / "raw_elcs_predictions.csv",
    index=False
)

print("Raw eLCS predictions saved.")
print(raw_results.head())

Raw eLCS predictions saved


**Creating the cleaned dataset**

In [8]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split


# ============================================================
# LOAD RAW DATA
# ============================================================

df_clean_source = pd.read_csv("../data/weatherAUS.csv")

print("Original shape:", df_clean_source.shape)


# ============================================================
# REMOVE MISSING TARGET
# RainTomorrow cannot be imputed because it is the class label
# ============================================================

df_clean_source = df_clean_source.dropna(
    subset=["RainTomorrow"]
).copy()

df_clean_source["RainTomorrow"] = (
    df_clean_source["RainTomorrow"]
    .astype(str)
    .str.strip()
    .map({
        "No": 0,
        "Yes": 1
    })
)

print("After removing missing target:", df_clean_source.shape)
print(df_clean_source["RainTomorrow"].value_counts())


# ============================================================
# SPLIT BEFORE LEARNED PREPROCESSING
# ============================================================

x_source = df_clean_source.drop(
    columns=["RainTomorrow"]
)

y_source = df_clean_source["RainTomorrow"].astype(int)

clean_x_train_raw, clean_x_test_raw, clean_y_train, clean_y_test = (
    train_test_split(
        x_source,
        y_source,
        test_size=0.20,
        random_state=42,
        stratify=y_source
    )
)

# separate copies so the original split remains available
clean_x_train = clean_x_train_raw.copy()
clean_x_test = clean_x_test_raw.copy()

print("\nTraining rows:", len(clean_x_train))
print("Testing rows:", len(clean_x_test))

print("\nTraining target:")
print(clean_y_train.value_counts())

print("\nTesting target:")
print(clean_y_test.value_counts())

Original shape: (145460, 23)
After removing missing target: (142193, 23)
RainTomorrow
0    110316
1     31877
Name: count, dtype: int64

Training rows: 113754
Testing rows: 28439

Training target:
RainTomorrow
0    88252
1    25502
Name: count, dtype: int64

Testing target:
RainTomorrow
0    22064
1     6375
Name: count, dtype: int64


In [9]:
# ============================================================
# CLEAN CATEGORY FORMATTING
# ============================================================

clean_x_train = clean_x_train.copy()
clean_x_test = clean_x_test.copy()

categorical_columns = [
    "Location",
    "WindGustDir",
    "WindDir9am",
    "WindDir3pm",
    "RainToday"
]

# strip accidental whitespace
for col in categorical_columns:
    clean_x_train[col] = clean_x_train[col].str.strip()
    clean_x_test[col] = clean_x_test[col].str.strip()


# ============================================================
# NUMERICAL IMPUTATION
# MEDIANS ARE LEARNED FROM TRAINING DATA ONLY
# ============================================================

numeric_columns = [
    "MinTemp",
    "MaxTemp",
    "Rainfall",
    "Evaporation",
    "Sunshine",
    "WindGustSpeed",
    "WindSpeed9am",
    "WindSpeed3pm",
    "Humidity9am",
    "Humidity3pm",
    "Pressure9am",
    "Pressure3pm",
    "Cloud9am",
    "Cloud3pm",
    "Temp9am",
    "Temp3pm"
]

train_medians = clean_x_train[numeric_columns].median()

for col in numeric_columns:
    clean_x_train[col] = clean_x_train[col].fillna(
        train_medians[col]
    )

    clean_x_test[col] = clean_x_test[col].fillna(
        train_medians[col]
    )


# ============================================================
# CATEGORICAL IMPUTATION
# MODES ARE LEARNED FROM TRAINING DATA ONLY
# ============================================================

categorical_fill_columns = [
    "WindGustDir",
    "WindDir9am",
    "WindDir3pm",
    "RainToday"
]

train_modes = {}

for col in categorical_fill_columns:
    train_modes[col] = clean_x_train[col].mode()[0]

    clean_x_train[col] = clean_x_train[col].fillna(
        train_modes[col]
    )

    clean_x_test[col] = clean_x_test[col].fillna(
        train_modes[col]
    )


# ============================================================
# CHECK RESULTS
# ============================================================

print("Training medians:")
print(train_medians)

print("\nTraining modes:")
print(train_modes)

print("\nMissing values in training:")
print(clean_x_train.isna().sum().sum())

print("\nMissing values in testing:")
print(clean_x_test.isna().sum().sum())

Training medians:
MinTemp            12.0
MaxTemp            22.6
Rainfall            0.0
Evaporation         4.8
Sunshine            8.5
WindGustSpeed      39.0
WindSpeed9am       13.0
WindSpeed3pm       19.0
Humidity9am        70.0
Humidity3pm        52.0
Pressure9am      1017.6
Pressure3pm      1015.2
Cloud9am            5.0
Cloud3pm            5.0
Temp9am            16.7
Temp3pm            21.1
dtype: float64

Training modes:
{'WindGustDir': 'W', 'WindDir9am': 'N', 'WindDir3pm': 'SE', 'RainToday': 'No'}

Missing values in training:
0

Missing values in testing:
0


In [10]:
# ============================================================
# APPLY ESTABLISHED DATA CLEANING RULES
# ============================================================

# Cloud cover is measured from 0 to 8 oktas.
# Cap previously identified invalid values above 8.
for col in ["Cloud9am", "Cloud3pm"]:
    clean_x_train.loc[
        clean_x_train[col] > 8,
        col
    ] = 8

    clean_x_test.loc[
        clean_x_test[col] > 8,
        col
    ] = 8


# Previously investigated Williamtown evaporation anomaly.
# Use the TRAINING evaporation median as the replacement value.
williamtown_train = (
    (clean_x_train["Location"] == "Williamtown")
    & (clean_x_train["Evaporation"] == 145.0)
)

williamtown_test = (
    (clean_x_test["Location"] == "Williamtown")
    & (clean_x_test["Evaporation"] == 145.0)
)

clean_x_train.loc[
    williamtown_train,
    "Evaporation"
] = train_medians["Evaporation"]

clean_x_test.loc[
    williamtown_test,
    "Evaporation"
] = train_medians["Evaporation"]

print(
    "Williamtown anomaly in train:",
    williamtown_train.sum()
)

print(
    "Williamtown anomaly in test:",
    williamtown_test.sum()
)

# ============================================================
# DATE FEATURES
# ============================================================

clean_x_train["Date"] = pd.to_datetime(
    clean_x_train["Date"]
)

clean_x_test["Date"] = pd.to_datetime(
    clean_x_test["Date"]
)

for data in [clean_x_train, clean_x_test]:
    data["Year"] = data["Date"].dt.year
    data["Month"] = data["Date"].dt.month
    data["Day"] = data["Date"].dt.day

    data.drop(
        columns=["Date"],
        inplace=True
    )


# ============================================================
# RAIN TODAY BINARY ENCODING
# ============================================================

rain_today_map = {
    "No": 0,
    "Yes": 1
}

clean_x_train["RainToday"] = (
    clean_x_train["RainToday"]
    .map(rain_today_map)
)

clean_x_test["RainToday"] = (
    clean_x_test["RainToday"]
    .map(rain_today_map)
)


# ============================================================
# WIND DIRECTION SINE/COSINE ENCODING
# ============================================================

wind_angles = {
    "N": 0,
    "NNE": 22.5,
    "NE": 45,
    "ENE": 67.5,
    "E": 90,
    "ESE": 112.5,
    "SE": 135,
    "SSE": 157.5,
    "S": 180,
    "SSW": 202.5,
    "SW": 225,
    "WSW": 247.5,
    "W": 270,
    "WNW": 292.5,
    "NW": 315,
    "NNW": 337.5
}

wind_columns = [
    "WindGustDir",
    "WindDir9am",
    "WindDir3pm"
]

for col in wind_columns:

    train_angle = clean_x_train[col].map(
        wind_angles
    )

    test_angle = clean_x_test[col].map(
        wind_angles
    )

    clean_x_train[col + "_sin"] = np.sin(
        np.radians(train_angle)
    )

    clean_x_train[col + "_cos"] = np.cos(
        np.radians(train_angle)
    )

    clean_x_test[col + "_sin"] = np.sin(
        np.radians(test_angle)
    )

    clean_x_test[col + "_cos"] = np.cos(
        np.radians(test_angle)
    )


clean_x_train = clean_x_train.drop(
    columns=wind_columns
)

clean_x_test = clean_x_test.drop(
    columns=wind_columns
)

Williamtown anomaly in train: 1
Williamtown anomaly in test: 0


In [11]:
# ============================================================
# LOCATION ENCODING
# FIT MAPPING ON TRAINING DATA ONLY
# ============================================================

train_locations = sorted(
    clean_x_train["Location"].unique()
)

location_map = {
    location: number
    for number, location in enumerate(train_locations)
}

clean_x_train["Location"] = (
    clean_x_train["Location"]
    .map(location_map)
)

clean_x_test["Location"] = (
    clean_x_test["Location"]
    .map(location_map)
)


# check whether test contained any unseen locations
print(
    "Unknown test locations:",
    clean_x_test["Location"].isna().sum()
)

print(
    "Number of training locations:",
    len(location_map)
)

Unknown test locations: 0
Number of training locations: 49


In [12]:
# ============================================================
# FINAL VALIDATION
# ============================================================

print("Train shape:", clean_x_train.shape)
print("Test shape:", clean_x_test.shape)

print(
    "\nTraining missing values:",
    clean_x_train.isna().sum().sum()
)

print(
    "Testing missing values:",
    clean_x_test.isna().sum().sum()
)

print(
    "\nNon-numeric training columns:",
    clean_x_train
    .select_dtypes(exclude="number")
    .columns
    .tolist()
)

print(
    "\nNon-numeric testing columns:",
    clean_x_test
    .select_dtypes(exclude="number")
    .columns
    .tolist()
)

print("\nColumns:")
print(clean_x_train.columns.tolist())

Train shape: (113754, 27)
Test shape: (28439, 27)

Training missing values: 0
Testing missing values: 0

Non-numeric training columns: []

Non-numeric testing columns: []

Columns:
['Location', 'MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation', 'Sunshine', 'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm', 'Temp9am', 'Temp3pm', 'RainToday', 'Year', 'Month', 'Day', 'WindGustDir_sin', 'WindGustDir_cos', 'WindDir9am_sin', 'WindDir9am_cos', 'WindDir3pm_sin', 'WindDir3pm_cos']


In [13]:
# ============================================================
# FINAL LEAKAGE-FREE DATASET CHECKS
# ============================================================

expected_columns = [
    "Location",
    "MinTemp",
    "MaxTemp",
    "Rainfall",
    "Evaporation",
    "Sunshine",
    "WindGustSpeed",
    "WindSpeed9am",
    "WindSpeed3pm",
    "Humidity9am",
    "Humidity3pm",
    "Pressure9am",
    "Pressure3pm",
    "Cloud9am",
    "Cloud3pm",
    "Temp9am",
    "Temp3pm",
    "RainToday",
    "Year",
    "Month",
    "Day",
    "WindGustDir_sin",
    "WindGustDir_cos",
    "WindDir9am_sin",
    "WindDir9am_cos",
    "WindDir3pm_sin",
    "WindDir3pm_cos"
]

print(
    "Training columns correct:",
    clean_x_train.columns.tolist() == expected_columns
)

print(
    "Testing columns correct:",
    clean_x_test.columns.tolist() == expected_columns
)

print(
    "Training X/Y indices aligned:",
    clean_x_train.index.equals(clean_y_train.index)
)

print(
    "Testing X/Y indices aligned:",
    clean_x_test.index.equals(clean_y_test.index)
)

print(
    "Training rows:",
    len(clean_x_train),
    len(clean_y_train)
)

print(
    "Testing rows:",
    len(clean_x_test),
    len(clean_y_test)
)

Training columns correct: True
Testing columns correct: True
Training X/Y indices aligned: True
Testing X/Y indices aligned: True
Training rows: 113754 113754
Testing rows: 28439 28439


In [14]:
# ============================================================
# LOCK FINAL LEAKAGE-FREE CLEANED DATASETS
# ============================================================

x_train_clean = clean_x_train.reset_index(drop=True).copy()
x_test_clean = clean_x_test.reset_index(drop=True).copy()

y_train_clean = clean_y_train.reset_index(drop=True).copy()
y_test_clean = clean_y_test.reset_index(drop=True).copy()

print("Final cleaned training shape:", x_train_clean.shape)
print("Final cleaned testing shape:", x_test_clean.shape)

print("\nTraining target:")
print(y_train_clean.value_counts())

print("\nTesting target:")
print(y_test_clean.value_counts())

Final cleaned training shape: (113754, 27)
Final cleaned testing shape: (28439, 27)

Training target:
RainTomorrow
0    88252
1    25502
Name: count, dtype: int64

Testing target:
RainTomorrow
0    22064
1     6375
Name: count, dtype: int64


In [15]:
from pathlib import Path

data_dir = Path("../data")
data_dir.mkdir(parents=True, exist_ok=True)

clean_train_export = x_train_clean.copy()
clean_train_export["RainTomorrow"] = y_train_clean.to_numpy()

clean_test_export = x_test_clean.copy()
clean_test_export["RainTomorrow"] = y_test_clean.to_numpy()

clean_train_export.to_csv(
    data_dir / "weatherAUS_clean_train.csv",
    index=False
)

clean_test_export.to_csv(
    data_dir / "weatherAUS_clean_test.csv",
    index=False
)

print("Saved leakage-free train and test datasets.")
print("Train export:", clean_train_export.shape)
print("Test export:", clean_test_export.shape)

Saved leakage-free train and test datasets.
Train export: (113754, 28)
Test export: (28439, 28)


**Cleaned eLCS**

In [20]:
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

from skeLCS.eLCS import eLCS

In [17]:
discrete_columns = [
    "Location",
    "RainToday"
]

discrete_attributes = np.array([
    x_train_clean.columns.get_loc(col)
    for col in discrete_columns
])

print("Discrete columns:")
print(discrete_columns)

print("\nDiscrete attribute indices:")
print(discrete_attributes)

print("\nTraining shape:")
print(x_train_clean.shape)

print("\nTesting shape:")
print(x_test_clean.shape)

Discrete columns:
['Location', 'RainToday']

Discrete attribute indices:
[ 0 17]

Training shape:
(113754, 27)

Testing shape:
(28439, 27)


In [21]:
clean_elcs = eLCS(
    learning_iterations=len(x_train_clean),
    N=10000,
    discrete_attribute_limit="d",
    specified_attributes=discrete_attributes,
    random_state=42
)

print("Learning iterations:", len(x_train_clean))
print("Population limit N:", 10000)

_ = clean_elcs.fit(
    x_train_clean.to_numpy(),
    y_train_clean.to_numpy()
)

print("Cleaned eLCS training complete.")

Learning iterations: 113754
Population limit N: 10000
Cleaned eLCS training complete.


In [19]:
clean_elcs_predictions = clean_elcs.predict(
    x_test_clean.to_numpy()
)

print("Prediction worked.")
print(len(clean_elcs_predictions))

Prediction worked.
28439


In [22]:
clean_elcs_predictions = clean_elcs.predict(
    x_test_clean.to_numpy()
)

clean_accuracy = accuracy_score(
    y_test_clean,
    clean_elcs_predictions
)

clean_balanced_accuracy = balanced_accuracy_score(
    y_test_clean,
    clean_elcs_predictions
)

print("Accuracy:", clean_accuracy)

print(
    "Balanced Accuracy:",
    clean_balanced_accuracy
)

print("\nClassification Report:")
print(
    classification_report(
        y_test_clean,
        clean_elcs_predictions
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test_clean,
        clean_elcs_predictions
    )
)

from pathlib import Path

results_dir = Path("../results")
results_dir.mkdir(
    parents=True,
    exist_ok=True
)

clean_results = pd.DataFrame({
    "row_id": clean_x_test.index.to_numpy(),
    "actual": y_test_clean.to_numpy(),
    "prediction": clean_elcs_predictions
})

clean_results.to_csv(
    results_dir / "clean_elcs_predictions.csv",
    index=False
)

print("Saved clean eLCS predictions.")
clean_elcs.export_final_rule_population(
    x_train_clean.columns.values,
    "RainTomorrow",
    filename=str(
        results_dir / "clean_elcs_rules.csv"
    ),
    DCAL=False
)

clean_elcs.export_iteration_tracking_data(
    str(
        results_dir / "clean_elcs_iterations.csv"
    )
)

Accuracy: 0.8225324378494321
Balanced Accuracy: 0.6197166993701033

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.99      0.90     22064
           1       0.85      0.25      0.39      6375

    accuracy                           0.82     28439
   macro avg       0.84      0.62      0.64     28439
weighted avg       0.83      0.82      0.78     28439


Confusion Matrix:
[[21785   279]
 [ 4768  1607]]
Saved clean eLCS predictions.


**Improved eLCS**

In [23]:
# ============================================================
# IMPROVED eLCS - 2:1 PARTIAL BALANCING
# ============================================================

improved_train = x_train_clean.copy()
improved_train["RainTomorrow"] = y_train_clean.to_numpy()

no_rain_train = improved_train[
    improved_train["RainTomorrow"] == 0
].copy()

rain_train = improved_train[
    improved_train["RainTomorrow"] == 1
].copy()

print("Original training distribution:")
print(improved_train["RainTomorrow"].value_counts())

# keep ALL rainy examples
rain_sample = rain_train.copy()

# keep twice as many No-rain examples
no_rain_sample = no_rain_train.sample(
    n=len(rain_sample) * 2,
    random_state=42
)

balanced_train = pd.concat(
    [
        no_rain_sample,
        rain_sample
    ],
    ignore_index=True
)

balanced_train = balanced_train.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

x_train_improved = balanced_train.drop(
    columns=["RainTomorrow"]
)

y_train_improved = balanced_train[
    "RainTomorrow"
].astype(int)

print("\nImproved training distribution:")
print(y_train_improved.value_counts())

print("\nImproved training shape:")
print(x_train_improved.shape)

print("\nUntouched test distribution:")
print(y_test_clean.value_counts())

Original training distribution:
RainTomorrow
0    88252
1    25502
Name: count, dtype: int64

Improved training distribution:
RainTomorrow
0    51004
1    25502
Name: count, dtype: int64

Improved training shape:
(76506, 27)

Untouched test distribution:
RainTomorrow
0    22064
1     6375
Name: count, dtype: int64


In [24]:
improved_discrete_attributes = np.array([
    x_train_improved.columns.get_loc("Location"),
    x_train_improved.columns.get_loc("RainToday")
])

improved_elcs = eLCS(
    learning_iterations=len(x_train_improved),
    N=10000,
    discrete_attribute_limit="d",
    specified_attributes=improved_discrete_attributes,
    random_state=42
)

print("Learning iterations:", len(x_train_improved))
print("N:", 10000)

Learning iterations: 76506
N: 10000


In [25]:
# ============================================================
# TRAIN IMPROVED eLCS
# ============================================================

_ = improved_elcs.fit(
    x_train_improved.to_numpy(),
    y_train_improved.to_numpy()
)

print("Improved eLCS training complete.")
# ============================================================
# PREDICT ON ORIGINAL TEST SET
# ============================================================

improved_elcs_predictions = improved_elcs.predict(
    x_test_clean.to_numpy()
)

print("Number of predictions:", len(improved_elcs_predictions))
print("Number of test rows:", len(y_test_clean))

Improved eLCS training complete.
Number of predictions: 28439
Number of test rows: 28439


In [26]:
# ============================================================
# EVALUATE IMPROVED eLCS
# ============================================================

improved_accuracy = accuracy_score(
    y_test_clean,
    improved_elcs_predictions
)

improved_balanced_accuracy = balanced_accuracy_score(
    y_test_clean,
    improved_elcs_predictions
)

print("Accuracy:", improved_accuracy)

print(
    "Balanced Accuracy:",
    improved_balanced_accuracy
)

print("\nClassification Report:")
print(
    classification_report(
        y_test_clean,
        improved_elcs_predictions
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test_clean,
        improved_elcs_predictions
    )
)

Accuracy: 0.8409578395864834
Balanced Accuracy: 0.7106731398143014

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.95      0.90     22064
           1       0.72      0.47      0.57      6375

    accuracy                           0.84     28439
   macro avg       0.79      0.71      0.74     28439
weighted avg       0.83      0.84      0.83     28439


Confusion Matrix:
[[20891  1173]
 [ 3350  3025]]


In [28]:
# ============================================================
# SAVE IMPROVED eLCS PREDICTIONS
# ============================================================

from pathlib import Path

results_dir = Path("../results")
results_dir.mkdir(
    parents=True,
    exist_ok=True
)

test_row_ids = clean_x_test.index.to_numpy()

assert len(test_row_ids) == len(y_test_clean)
assert len(test_row_ids) == len(improved_elcs_predictions)

improved_results = pd.DataFrame({
    "row_id": test_row_ids,
    "actual": y_test_clean.to_numpy(),
    "prediction": improved_elcs_predictions
})

improved_results.to_csv(
    results_dir / "improved_elcs_predictions.csv",
    index=False
)

print("Saved improved eLCS predictions.")
print(improved_results.head())

# ============================================================
# EXPORT FINAL RULE POPULATION
# ============================================================

improved_elcs.export_final_rule_population(
    x_train_improved.columns.values,
    "RainTomorrow",
    filename=str(
        results_dir / "improved_elcs_rules.csv"
    ),
    DCAL=False
)

print("Saved improved eLCS rules.")
# ============================================================
# EXPORT ITERATION TRACKING
# ============================================================

improved_elcs.export_iteration_tracking_data(
    str(
        results_dir / "improved_elcs_iterations.csv"
    )
)

print("Saved improved eLCS iteration data.")

# ============================================================
# FINAL IMPROVED eLCS CHECK
# ============================================================

print(
    "Predictions file exists:",
    (
        results_dir
        / "improved_elcs_predictions.csv"
    ).exists()
)

print(
    "Rules file exists:",
    (
        results_dir
        / "improved_elcs_rules.csv"
    ).exists()
)

print(
    "Iterations file exists:",
    (
        results_dir
        / "improved_elcs_iterations.csv"
    ).exists()
)

Saved improved eLCS predictions.
   row_id  actual  prediction
0   48872       0           0
1  100864       1           1
2  123233       0           0
3    7228       0           0
4    3948       0           0
Saved improved eLCS rules.
Saved improved eLCS iteration data.
Predictions file exists: True
Rules file exists: True
Iterations file exists: True


**Random Forest**


In [29]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

df = pd.read_csv("../data/weatherAUS.csv")

# remove rows where the TARGET is missing
df = df.dropna(subset=["RainTomorrow"]).copy()

# only map if target is still Yes/No
df["RainTomorrow"] = df["RainTomorrow"].map({
    "No": 0,
    "Yes": 1
}).astype(int)
print(df["RainTomorrow"].value_counts())

# Date is not being used as a predictor
df = df.drop(columns=["Date"], errors="ignore")

print(df.shape)
print(df["RainTomorrow"].value_counts(dropna=False))

RainTomorrow
0    110316
1     31877
Name: count, dtype: int64
(142193, 22)
RainTomorrow
0    110316
1     31877
Name: count, dtype: int64


In [30]:
x = df.drop(columns=["RainTomorrow"])
y = df["RainTomorrow"]

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", x_train.shape)
print("Testing:", x_test.shape)
print(y_test.value_counts())

Training: (113754, 21)
Testing: (28439, 21)
RainTomorrow
0    22064
1     6375
Name: count, dtype: int64


In [31]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

numeric_columns = x_train.select_dtypes(exclude=["object"]).columns
categorical_columns = x_train.select_dtypes(include=["object"]).columns

numeric_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_preprocessor, numeric_columns),
        ("categorical", categorical_preprocessor, categorical_columns)
    ]
)

print("Numerical columns:", len(numeric_columns))
print("Categorical columns:", len(categorical_columns))

Numerical columns: 16
Categorical columns: 5


C:\Users\JaskaranSingh\AppData\Local\Temp\ipykernel_640\3612860163.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = x_train.select_dtypes(include=["object"]).columns


In [32]:
rf_model_tuned2 = RandomForestClassifier(
    n_estimators=300,
    max_depth=25,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_pipeline_tuned2 = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("random_forest", rf_model_tuned2)
    ]
)

rf_pipeline_tuned2.fit(x_train, y_train)

train_predictions_tuned2 = rf_pipeline_tuned2.predict(x_train)
test_predictions_tuned2 = rf_pipeline_tuned2.predict(x_test)

print(
    "Training Accuracy:",
    accuracy_score(y_train, train_predictions_tuned2)
)

print(
    "Testing Accuracy:",
    accuracy_score(y_test, test_predictions_tuned2)
)

print(
    "Balanced Accuracy:",
    balanced_accuracy_score(y_test, test_predictions_tuned2)
)

print("\nClassification Report:")
print(classification_report(y_test, test_predictions_tuned2))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_predictions_tuned2))

Training Accuracy: 0.9438613147669532
Testing Accuracy: 0.8342065473469531
Balanced Accuracy: 0.7951080386469309

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.87      0.89     22064
           1       0.61      0.72      0.66      6375

    accuracy                           0.83     28439
   macro avg       0.76      0.80      0.78     28439
weighted avg       0.85      0.83      0.84     28439


Confusion Matrix:
[[19107  2957]
 [ 1758  4617]]


In [33]:
rf_tuned_results = pd.DataFrame({
    "actual": y_test.to_numpy(),
    "prediction": test_predictions_tuned2
})

rf_tuned_results.to_csv(
    "../results/random_forest_tuned_predictions.csv",
    index=False
)

print("Saved.")

Saved.


**XGBoost**

In [34]:
# Install packages into the notebook's current Python environment
%pip install pandas numpy scikit-learn xgboost -q

# Imports
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from xgboost import XGBClassifier

Note: you may need to restart the kernel to use updated packages.


In [35]:
y_train = y_train.replace({
    "No": 0,
    "Yes": 1
}).astype(int)

y_test = y_test.replace({
    "No": 0,
    "Yes": 1
}).astype(int)

print("Training target:")
print(y_train.value_counts())

print("\nTesting target:")
print(y_test.value_counts())

Training target:
RainTomorrow
0    88252
1    25502
Name: count, dtype: int64

Testing target:
RainTomorrow
0    22064
1     6375
Name: count, dtype: int64


In [36]:
xgb_model = XGBClassifier(
    n_estimators=5000,
    learning_rate=1,
    max_depth=10,
    scale_pos_weight=3.65,
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss"
)

xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("xgboost", xgb_model)
])

xgb_pipeline.fit(x_train, y_train)

xgb_predictions = xgb_pipeline.predict(x_test)
print(
    "Balanced Accuracy:",
    balanced_accuracy_score(y_test, xgb_predictions)
)

Balanced Accuracy: 0.7629707446430349


In [37]:
xgb_results = pd.DataFrame({
    "actual": y_test.to_numpy(),
    "prediction": xgb_predictions
})

xgb_results.to_csv(
    "../results/xgboost_predictions.csv",
    index=False
)

**Linear Regression**


In [38]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

lr_model = Pipeline([
    ("preprocessor", preprocessor),
    ("scaler", StandardScaler(with_mean=False)),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    ))
])

lr_model.fit(x_train, y_train)

lr_predictions = lr_model.predict(x_test)

lr_model.fit(x_train, y_train)

lr_predictions = lr_model.predict(x_test)

print("Accuracy:", accuracy_score(y_test, lr_predictions))

print("\nClassification Report:")
print(classification_report(y_test, lr_predictions))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, lr_predictions))

Accuracy: 0.7941559126551566

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.80      0.86     22064
           1       0.53      0.78      0.63      6375

    accuracy                           0.79     28439
   macro avg       0.73      0.79      0.74     28439
weighted avg       0.84      0.79      0.81     28439


Confusion Matrix:
[[17637  4427]
 [ 1427  4948]]
